# YOLO – Pascal VOC Object Detection

This project uses a pretrained YOLO detector for multi-class object detection on the Pascal VOC dataset. The notebook covers dataset preparation, transfer learning, training, evaluation, class-wise analysis, prediction visualization, confidence analysis, and model export.

## 1. Import Libraries and Check Runtime

The notebook installs Ultralytics and checks the PyTorch, CUDA, and GPU environment before training.

In [1]:
!pip -q install -U ultralytics

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import ultralytics
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset
from IPython.display import display

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 8.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics: 8.4.163
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## 2. Configure Training

The main settings define the model, input resolution, batch size, training duration, and runtime device.

In [2]:
SEED = 42
MODEL_NAME = "yolo26n.pt"
DATASET_YAML = "VOC.yaml"

EPOCHS = 30
IMAGE_SIZE = 640
BATCH_SIZE = 16
WORKERS = 2
DEVICE = 0 if torch.cuda.is_available() else "cpu"

PROJECT_DIR = Path("/content/yolo_voc_runs")
TRAIN_RUN_NAME = "train"

np.random.seed(SEED)
torch.manual_seed(SEED)

print("Model:", MODEL_NAME)
print("Dataset:", DATASET_YAML)
print("Epochs:", EPOCHS)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Device:", DEVICE)

Model: yolo26n.pt
Dataset: VOC.yaml
Epochs: 30
Image size: 640
Batch size: 16
Device: 0


## 3. Load the Pascal VOC Dataset

The Ultralytics VOC configuration downloads the dataset and converts the original Pascal VOC bounding-box annotations to YOLO format.

In [3]:
dataset_info = check_det_dataset(DATASET_YAML)
dataset_path = Path(dataset_info["path"])

raw_names = dataset_info["names"]
class_names = list(raw_names.values()) if isinstance(raw_names, dict) else list(raw_names)

print("Dataset path:", dataset_path)
print("Number of classes:", len(class_names))
print("Classes:", class_names)


WARNING ⚠️ Dataset 'VOC.yaml' images not found, missing path '/content/datasets/VOC/images/test2007'
Unzipping /content/datasets/VOC/images/VOCtrainval_06-Nov-2007.zip to /content/datasets/VOC/images/VOCdevkit...: 100% ━━━━━━━━━━━━ 10945/10945 576.6files/s 19.0s
Unzipping /content/datasets/VOC/images/VOCtest_06-Nov-2007.zip to /content/datasets/VOC/images/VOCdevkit...: 100% ━━━━━━━━━━━━ 10357/10357 648.4files/s 16.0s
Unzipping /content/datasets/VOC/images/VOCtrainval_11-May-2012.zip to /content/datasets/VOC/images/VOCdevkit...: 100% ━━━━━━━━━━━━ 40189/40189 1.5Kfiles/s 26.6s
train2012: 100% ━━━━━━━━━━━━ 5717/5717 3.3Kit/s 1.7s
val2012: 100% ━━━━━━━━━━━━ 5823/5823 2.7Kit/s 2.2s
train2007: 100% ━━━━━━━━━━━━ 2501/2501 1.4Kit/s 1.8s
val2007: 100% ━━━━━━━━━━━━ 2510/2510 2.2Kit/s 1.2s
test2007: 100% ━━━━━━━━━━━━ 4952/4952 3.2Kit/s 1.6s
Dataset download success ✅ (67.2s), saved to /content/datasets

Dataset path: /content/datasets/VOC
Number of classes: 20
Classes: ['aeroplane', 'bicycle', '

## 4. Inspect the Detection Dataset

The dataset configuration is checked to confirm the training and evaluation image locations and counts.

In [4]:
def resolve_split_paths(split_value):
    values = split_value if isinstance(split_value, list) else [split_value]
    resolved = []
    for value in values:
        path = Path(value)
        if not path.is_absolute():
            path = dataset_path / path
        resolved.append(path)
    return resolved

train_paths = resolve_split_paths(dataset_info["train"])
val_paths = resolve_split_paths(dataset_info["val"])

train_image_count = sum(len(list(path.glob("*.jpg"))) for path in train_paths)
val_image_count = sum(len(list(path.glob("*.jpg"))) for path in val_paths)

print("Training image count:", train_image_count)
print("Evaluation image count:", val_image_count)
print("Training directories:")
for path in train_paths:
    print(path)
print("Evaluation directory:")
for path in val_paths:
    print(path)

Training image count: 16551
Evaluation image count: 4952
Training directories:
/content/datasets/VOC/images/train2012
/content/datasets/VOC/images/train2007
/content/datasets/VOC/images/val2012
/content/datasets/VOC/images/val2007
Evaluation directory:
/content/datasets/VOC/images/test2007


## 5. Load the Pretrained YOLO Model

A pretrained YOLO26n detection model is loaded as the starting point for transfer learning on Pascal VOC.

In [5]:
model = YOLO(MODEL_NAME)
model.info()

YOLO26n summary: 260 layers, 2,572,280 parameters, 0 gradients, 6.2 GFLOPs


(260, 2572280, 0, 6.2396672)

## 6. Train the YOLO Detector

The pretrained detector is fine-tuned on the Pascal VOC training images for the configured number of epochs.

In [ ]:
train_results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    project=str(PROJECT_DIR),
    name=TRAIN_RUN_NAME,
    exist_ok=True,
    pretrained=True,
    optimizer="auto",
    patience=8,
    plots=True,
    save=True,
    seed=SEED,
    verbose=True
)

TRAIN_RUN = PROJECT_DIR / TRAIN_RUN_NAME
print("Training complete.")
print("Training run:", TRAIN_RUN)

Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=VOC.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto,

## 7. Inspect Training History

The YOLO training log is loaded to review losses and detection metrics for each epoch.

In [ ]:
results_csv = TRAIN_RUN / "results.csv"

if not results_csv.exists():
    raise FileNotFoundError(f"Training history not found at {results_csv}")

history = pd.read_csv(results_csv)
history.columns = [column.strip() for column in history.columns]

display(history.tail())

## 8. Plot Training and Validation Metrics

The learning curves show how detection losses and validation metrics change during training.

In [ ]:
loss_columns = [
    column for column in [
        "train/box_loss",
        "train/cls_loss",
        "train/dfl_loss",
        "val/box_loss",
        "val/cls_loss",
        "val/dfl_loss"
    ]
    if column in history.columns
]

metric_columns = [
    column for column in [
        "metrics/precision(B)",
        "metrics/recall(B)",
        "metrics/mAP50(B)",
        "metrics/mAP50-95(B)"
    ]
    if column in history.columns
]

if not metric_columns:
    raise KeyError("No validation detection metrics were found in results.csv")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for column in loss_columns:
    axes[0].plot(history["epoch"], history[column], label=column)

axes[0].set_title("Detection Losses")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

for column in metric_columns:
    axes[1].plot(history["epoch"], history[column], label=column)

axes[1].set_title("Validation Detection Metrics")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Metric")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Load the Best Checkpoint

The saved checkpoint selected by the training run is loaded for final evaluation and prediction.

In [ ]:
best_model_path = TRAIN_RUN / "weights" / "best.pt"

if not best_model_path.exists():
    raise FileNotFoundError(f"Best model not found at {best_model_path}")

best_model = YOLO(str(best_model_path))
best_names = list(best_model.names.values()) if isinstance(best_model.names, dict) else list(best_model.names)

print("Best checkpoint:", best_model_path)
print("Number of classes:", len(best_names))
print("Classes:", best_names)

## 10. Evaluate the Best Detector

The best checkpoint is evaluated on the VOC2007 public evaluation split used by the Ultralytics VOC configuration.

In [ ]:
validation = best_model.val(
    data=DATASET_YAML,
    split="val",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    plots=True,
    verbose=True
)

print("Precision:", round(float(validation.box.mp), 4))
print("Recall:", round(float(validation.box.mr), 4))
print("mAP@0.50:", round(float(validation.box.map50), 4))
print("mAP@0.50:0.95:", round(float(validation.box.map), 4))

## 11. Summarize Detection Metrics

The final precision, recall, and mAP values are collected into one place for reporting.

In [ ]:
required_columns = [
    "metrics/precision(B)",
    "metrics/recall(B)",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)"
]

missing_columns = [column for column in required_columns if column not in history.columns]
if missing_columns:
    raise KeyError(f"Missing training metric columns: {missing_columns}")

best_map_epoch_index = history["metrics/mAP50-95(B)"].idxmax()
best_map_epoch = int(history.loc[best_map_epoch_index, "epoch"])

metrics_table = pd.DataFrame({
    "Metric": [
        "Precision",
        "Recall",
        "mAP@0.50",
        "mAP@0.50:0.95",
        "Highest logged mAP@0.50:0.95 Epoch"
    ],
    "Value": [
        float(validation.box.mp),
        float(validation.box.mr),
        float(validation.box.map50),
        float(validation.box.map),
        best_map_epoch
    ]
})

display(metrics_table)

## 12. Analyze Class-wise Performance

Per-class mAP shows how detection performance varies across the Pascal VOC categories.

In [ ]:
class_map = np.asarray(validation.box.maps, dtype=float)

if class_map.size != len(best_names):
    raise ValueError("The number of class metrics does not match the number of class names.")

class_metrics = pd.DataFrame({
    "Class": best_names,
    "mAP@0.50:0.95": class_map
}).sort_values("mAP@0.50:0.95", ascending=False).reset_index(drop=True)

display(class_metrics)

plt.figure(figsize=(11, 7))
plt.barh(class_metrics["Class"], class_metrics["mAP@0.50:0.95"])
plt.xlabel("mAP@0.50:0.95")
plt.ylabel("Class")
plt.title("Class-wise Detection Performance")
plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Select Evaluation Images

A small set of images from the evaluation split is selected for qualitative inspection.

In [ ]:
evaluation_images = []
for path in val_paths:
    evaluation_images.extend(sorted(path.glob("*.jpg")))

evaluation_images = sorted(evaluation_images)

if not evaluation_images:
    raise FileNotFoundError("No evaluation images were found in the Pascal VOC dataset.")

sample_count = min(6, len(evaluation_images))
sample_positions = np.linspace(0, len(evaluation_images) - 1, sample_count, dtype=int)
sample_images = [evaluation_images[position] for position in sample_positions]

print("Selected images:")
for image_path in sample_images:
    print(image_path)

## 14. Run Object Detection

The trained detector produces bounding boxes, class labels, and confidence scores for the selected images.

In [ ]:
prediction_results = best_model.predict(
    source=[str(path) for path in sample_images],
    imgsz=IMAGE_SIZE,
    conf=0.25,
    iou=0.7,
    device=DEVICE,
    verbose=False
)

print("Predictions generated:", len(prediction_results))

## 15. Visualize Predicted Bounding Boxes

The detections are shown on the input images so the predicted locations and labels can be inspected visually.

In [ ]:
fig, axes = plt.subplots(
    len(prediction_results),
    1,
    figsize=(12, 5 * len(prediction_results))
)

if len(prediction_results) == 1:
    axes = [axes]

for axis, result, image_path in zip(axes, prediction_results, sample_images):
    plotted = result.plot()
    axis.imshow(plotted[..., ::-1])
    axis.set_title(Path(image_path).name)
    axis.axis("off")

plt.tight_layout()
plt.show()

## 16. Summarize Sample Detections

The number of detected objects in each selected image is summarized before confidence analysis.

In [ ]:
detection_summary = []

for image_path, result in zip(sample_images, prediction_results):
    detection_count = len(result.boxes) if result.boxes is not None else 0
    detection_summary.append({
        "Image": Path(image_path).name,
        "Detections": int(detection_count)
    })

detection_summary_df = pd.DataFrame(detection_summary)
display(detection_summary_df)

## 17. Inspect Detection Confidence

The confidence distribution summarizes how strongly the model scores the detections in the selected images.

In [ ]:
confidence_values = []

for result in prediction_results:
    if result.boxes is not None and len(result.boxes) > 0:
        confidence_values.extend(
            result.boxes.conf.detach().cpu().numpy().tolist()
        )

confidence_values = np.asarray(confidence_values, dtype=float)

if confidence_values.size == 0:
    print("No detections were returned for the selected images.")
else:
    confidence_summary = pd.DataFrame({
        "Statistic": [
            "Detections",
            "Mean confidence",
            "Median confidence",
            "Minimum confidence",
            "Maximum confidence"
        ],
        "Value": [
            int(confidence_values.size),
            float(confidence_values.mean()),
            float(np.median(confidence_values)),
            float(confidence_values.min()),
            float(confidence_values.max())
        ]
    })

    display(confidence_summary)

    plt.figure(figsize=(9, 5))
    plt.hist(confidence_values, bins=15)
    plt.xlabel("Confidence")
    plt.ylabel("Number of Detections")
    plt.title("Prediction Confidence Distribution")
    plt.grid(alpha=0.3)
    plt.show()

## 18. Export the Trained Model

The best detector is exported to ONNX so it can be used outside the notebook environment.

In [ ]:
export_path = best_model.export(format="onnx")
print("Exported model:", export_path)

## 19. Key Findings



## 20. Conclusion

